# 03 · Limpieza y unión de fuentes

Lee los datos guardados en `data/raw/` (no hace peticiones nuevas), los limpia con las funciones de `src/cleaning.py` y guarda:

- `data/clean/municipios.csv`: una fila por municipio (pública, sin contactos).
- `data/clean/leads_organizaciones.csv`: organizaciones con contactos (interna, fuera de GitHub).
- `data/interno/contratos_ropa_region.csv`: contratos públicos de ropa de la región, con proveedores (interna, fuera de GitHub).
- `data/interno/competencia_osm.csv`, `data/interno/graficas_osm.csv` y `data/clean/leads_empresas.csv`: competencia y empresas de OpenStreetMap (internas, fuera de GitHub).

> **Cómo leer este notebook.** Antes de cada celda de código hay un bloque **Cómo funciona**: qué hace la celda, por qué se hace así y qué herramientas usa. Cuando algo no se ha visto en clase, se marca como **Nuevo** y se explica con más calma. Las funciones que se llaman (`recoleccion.…`, `cleaning.…`, `analisis.…`) están en la carpeta `src/`, con su explicación en el docstring.

**Cómo funciona**

- **Qué hace:** prepara el notebook: rutas, configuración y los módulos `recoleccion` (leer lo guardado) y `cleaning` (limpiar).
- **Por qué así:** este notebook no pide nada a internet: solo lee `data/raw/`. Así la limpieza se puede repetir siempre igual.
- **Herramientas:** `import`, `pathlib.Path`, list comprehension.
- **Nuevo (no visto en clase):** `importlib.reload` (recarga `src/` sin reiniciar el kernel).

In [109]:
import importlib
import sys
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

from src import cleaning, recoleccion

# Recarga el código de src/ por si ha cambiado (así no hace falta reiniciar el kernel)
importlib.reload(recoleccion)
importlib.reload(cleaning)

config = recoleccion.cargar_config(RAIZ / "config.yaml")
RUTA_RAW = RAIZ / config["rutas"]["raw"]
RUTA_CLEAN = RAIZ / config["rutas"]["clean"]
municipios = [m for lista in config["region"]["distritos"].values() for m in lista]
ine = config["ine"]

## 1. OpenStreetMap

**Cómo funciona**

- **Qué hace:** lee las respuestas guardadas de OpenStreetMap y las limpia con `cleaning.limpiar_osm`, que devuelve la tabla y un registro de cuántas filas quita cada paso (1.824 → 1.552).
- **Por qué así:** `limpiar_osm` hace los pasos en orden: 1) quita las filas sin nombre; 2) asigna el segmento; 3) pasa a club las instalaciones deportivas cuyo nombre es de club; 4) normaliza el nombre (minúsculas, sin acentos, «G.D.» → «grupo desportivo»); 5) da formato a email, web y teléfono; 6) quita duplicados y se queda con la fila con más contactos; 7) saca el año de fundación y el nivel de contacto; 8) añade fuente, licencia y fecha. Al final, `comprobar_osm` verifica con `assert` que no quedan nombres vacíos ni duplicados y que las coordenadas están dentro de la región.
- **Herramientas:** `notna`, `apply`, `str.contains`, `drop_duplicates`, `sort_values`, `assert`.
- **Nuevo (no visto en clase):** `normalize("NFKD")` para quitar acentos y expresiones regulares (`re`) para las abreviaturas.

In [110]:
# Lee las respuestas guardadas (solo_cache=True: no pide nada a la API)
resultados_osm = recoleccion.recoger_osm_region(
    municipios, config["overpass"], config["overpass"]["etiquetas"], RUTA_RAW, solo_cache=True
)

# Fecha de recogida: la del archivo guardado más reciente
archivos = list((RUTA_RAW / "overpass").glob("*.json"))
fecha_recogida = date.fromtimestamp(max(a.stat().st_mtime for a in archivos)).isoformat()

df_osm, registro = cleaning.limpiar_osm(resultados_osm, config["limpieza"]["licencia_osm"], fecha_recogida)
pd.Series(registro, name="filas")

overpass: 66 de 66 municipios leídos de los archivos guardados


filas_iniciales                 1824
sin_nombre_eliminadas            235
instalaciones_que_son_clubes      22
duplicados_eliminados             37
filas_finales                   1552
Name: filas, dtype: int64

**Cómo funciona**

- **Qué hace:** añade la freguesia a las organizaciones de los municipios del piloto.
- **Por qué así:** `freguesias_a_dataframe` recorre la respuesta: cada elemento de tipo «area» es una freguesia y lo que viene detrás pertenece a ella. Después se une con `merge(how="left")`: todas las organizaciones se quedan, tengan freguesia o no. El `assert` comprueba que la unión no ha duplicado filas.
- **Herramientas:** `merge(how="left")`, `drop_duplicates`, `assert`.

In [111]:
# Freguesia de cada organización en los municipios del piloto (solo archivos guardados)
resultados_freguesias = recoleccion.recoger_osm_region(
    config["region"]["piloto"], config["overpass"], config["overpass"]["etiquetas"], RUTA_RAW,
    solo_cache=True, por_freguesia=True,
)
df_osm = cleaning.anadir_freguesia(df_osm, resultados_freguesias)
print("Organizaciones con freguesia:", df_osm["freguesia"].notna().sum())

overpass_freguesias: 6 de 6 municipios leídos de los archivos guardados
Organizaciones con freguesia: 153


**Cómo funciona**

- **Qué hace:** cuenta las organizaciones de cada segmento.
- **Por qué así:** es la primera foto del mercado: los colegios son más de la mitad; clubes y asociaciones, muy pocos.
- **Herramientas:** `value_counts`.

In [112]:
# Organizaciones por segmento
df_osm["segmento"].value_counts()

segmento
escuela                  809
autarquia                304
instalacion_deportiva    189
bomberos                 108
asociacion                81
club                      61
Name: count, dtype: int64

**Cómo funciona**

- **Qué hace:** calcula el % de organizaciones según el mejor contacto público que tienen (email, solo web, solo teléfono o ninguno).
- **Por qué así:** es la primera pista para la H3. `normalize=True` da proporciones en vez de recuentos.
- **Herramientas:** `value_counts(normalize=True)`, `round`.

In [113]:
# Primera pista para la H3: qué contacto público tienen (en %)
(df_osm["nivel_contacto"].value_counts(normalize=True) * 100).round(1)

nivel_contacto
sin contacto     47.8
email            45.0
solo web          3.7
solo telefono     3.4
Name: proportion, dtype: float64

**Cómo funciona**

- **Qué hace:** cuenta cuántas organizaciones tienen año de fundación.
- **Por qué así:** la idea era usar los aniversarios como motivo de contacto; con tan pocos datos no sirve y se descarta (está en los límites del README).
- **Herramientas:** `notna().sum()`.

In [114]:
# Cobertura del año de fundación (para los aniversarios)
print("Con año de fundación:", df_osm["ano_fundacion"].notna().sum(), "de", len(df_osm))

Con año de fundación: 34 de 1552



### Juntas de freguesia que faltan en OpenStreetMap (piloto)

Cada freguesia tiene una junta, pero OpenStreetMap solo registra una parte. Con los límites de las freguesias del piloto se añaden las que faltan, sin contacto (hay que completarlo a mano). Las cifras de arriba (segmentos, contactos) son solo de OpenStreetMap; las juntas añadidas van a la lista de contactos, pero no a la tabla por municipio, para que todos los municipios se cuenten igual.


**Cómo funciona**

- **Qué hace:** añade las juntas de freguesia del piloto que no están en OpenStreetMap (sin contacto) y muestra la cobertura por municipio.
- **Por qué así:** cada freguesia tiene una junta, así que la lista completa sale de los límites de las freguesias. Una freguesia cuenta como cubierta si dentro hay una autarquía cuyo nombre dice «junta» o «união». Las añadidas llevan `origen = "límite de freguesia"` para completarlas a mano. Es una imputación justificada: que la junta existe es seguro; su contacto queda vacío, no se inventa.
- **Herramientas:** `merge`, `notna`, list comprehension, `pd.concat`, `groupby().agg()`.

In [115]:

df_osm, cobertura_juntas = cleaning.completar_juntas(df_osm, resultados_freguesias)
print("Juntas en OpenStreetMap:", cobertura_juntas["juntas_en_osm"].sum(), "de", cobertura_juntas["freguesias"].sum())
cobertura_juntas


Juntas en OpenStreetMap: 33 de 78


,municipio,freguesias,juntas_en_osm,juntas_anadidas
0,Arganil,14,12,2
1,Carregal do Sal,5,3,2
2,Nelas,7,5,2
3,Oliveira do Hospital,17,6,11
4,Seia,24,3,21
5,Tábua,11,4,7


**Cómo funciona**

- **Qué hace:** calcula la distancia en línea recta de cada organización a Oliveira do Hospital y muestra qué parte está a 20, 40 y 60 km.
- **Por qué así:** una marca pequeña vende primero cerca de casa; la distancia entra después en el índice y en la puntuación.
- **Herramientas:** `numpy` (calcula con la columna entera a la vez), f-strings con formato `:.0%`.
- **Nuevo (no visto en clase):** la fórmula del haversine: distancia sobre la esfera de la Tierra a partir de latitud y longitud.

In [116]:
# Distancia (km, en línea recta) de cada organización a la base de la marca (Oliveira do Hospital).
# Una marca pequeña vende primero cerca: la distancia entra después en la puntuación
df_osm = cleaning.anadir_distancia(df_osm, config["negocio"]["base"])
for radio in [20, 40, 60]:
    print(f"A {radio} km o menos: {(df_osm['km_a_la_base'] <= radio).mean():.0%} de las organizaciones")

A 20 km o menos: 8% de las organizaciones
A 40 km o menos: 34% de las organizaciones
A 60 km o menos: 61% de las organizaciones


## 2. INE: población por municipio

**Cómo funciona**

- **Qué hace:** limpia la población del INE (una fila por municipio) y le añade los jóvenes de 5 a 24 años y los mayores de 65.
- **Por qué así:** `ine_a_dataframe` convierte el JSON en tabla; `quedarse_con_totales` se queda con «HM» (hombres y mujeres) y el total; `valores_por_municipio` une por el nombre normalizado, porque el INE escribe los nombres con otras mayúsculas o acentos. Si un nombre existe como municipio y como región, se queda con el código más largo (7 cifras = municipio). El `assert` para si falta algún municipio.
- **Herramientas:** `merge`, `isin`, `groupby().sum()`, `sort_values`, `drop_duplicates`.

In [117]:
ruta_pob = RUTA_RAW / f"ine_{ine['indicador_poblacion']}_{ine['periodo']}.json"

if ruta_pob.exists():
    poblacion = cleaning.limpiar_poblacion(recoleccion.leer_json(ruta_pob), municipios)
    # Jóvenes (5 a 24 años) y mayores (65 o más): el mismo archivo trae los grupos de edad
    poblacion = poblacion.merge(cleaning.poblacion_por_edad(recoleccion.leer_json(ruta_pob), municipios), on="municipio")
    display(poblacion.sort_values("poblacion", ascending=False).head())
else:
    poblacion = None
    print("Falta el archivo del INE: ejecuta primero el notebook 02.")

,municipio,codigo_ine,poblacion,poblacion_5_24,poblacion_65_mas
2,Coimbra,16E0603,154224,29148,38233
39,Viseu,16G1823,106199,21652,24897
4,Figueira da Foz,16E0605,64468,11484,17565
56,Castelo Branco,16H0502,57148,10551,15718
57,Covilhã,16J0503,50165,8974,14558


## 3. INE: empresas del sector (competencia)

El INE no publica estas actividades (subclases CAE) por municipio, solo por región NUTS III. Por eso la competencia se mide por región, y cada municipio hereda la de su región.

**Cómo funciona**

- **Qué hace:** calcula la competencia (empresas de confección, bordado, impresión y comercio de ropa y deporte) por región NUTS III y por 10.000 habitantes.
- **Por qué así:** el INE no publica estas actividades (5 cifras) por municipio, solo por región; cada municipio hereda el dato de su región. Es una aproximación y se dice. Las 3 primeras cifras del código INE de un municipio son su región.
- **Herramientas:** dict comprehension, `pd.concat(axis=1)`, `join`, `str[:3]`.
- **Nuevo (no visto en clase):** las regiones NUTS III (la división estadística europea).

In [118]:
carpeta_empresas = RUTA_RAW / f"ine_{ine['indicador_empresas']}"
respuestas = {
    ruta.stem.split("_")[-1]: recoleccion.leer_json(ruta)
    for ruta in sorted(carpeta_empresas.glob(f"{ine['periodo_empresas']}_*.json"))
}
print("Actividades CAE disponibles:", list(respuestas))

if respuestas and poblacion is not None:
    regiones = sorted(poblacion["codigo_ine"].str[:3].unique())
    empresas = cleaning.empresas_por_nuts3(respuestas, recoleccion.leer_json(ruta_pob), regiones)
    display(empresas.sort_values("empresas_por_10000_hab", ascending=False))
else:
    empresas = None
    print("Faltan los datos de empresas o de población del INE: ejecuta primero el notebook 02.")

Actividades CAE disponibles: ['13991', '14120', '14131', '14190', '18120', '47640', '47711']


,nuts3,region_nuts3,poblacion_nuts3,cae_13991,cae_14120,cae_14131,cae_14190,cae_18120,cae_47640,cae_47711,empresas_sector,empresas_por_10000_hab
5,11C,Tâmega e Sousa,412284,45,13,533,82,56,39,510,1278,31.00
4,16G,Viseu Dão Lafões,261879,5,5,22,11,30,27,216,316,12.07
1,11D,Douro,183338,3,2,3,7,20,25,152,212,11.56
2,16J,Beiras e Serra da Estrela,212082,9,3,17,17,17,20,162,245,11.55
3,16I,Médio Tejo,238931,12,4,10,12,37,43,152,270,11.30
6,16E,Região de Coimbra,461013,10,7,40,23,60,52,317,509,11.04
0,16H,Beira Baixa,84717,7,0,7,2,4,5,52,77,9.09


**Cómo funciona**

- **Qué hace:** lee las empresas por municipio de 7 divisiones del INE (impresión, vestuario, alojamiento, restauración, salud y apoyo social), de 2022.
- **Por qué así:** estas divisiones (2 cifras) sí se publican por municipio, aunque un año antes. Impresión y vestuario miden la competencia en el índice; las otras, el mercado de la oferta Trabajo. Un municipio con el dato protegido por el INE queda vacío (NA), no a 0.
- **Herramientas:** dict comprehension, `glob`, `merge`, `isna().any(axis=1)`.

In [119]:
# Gráficas y confección por municipio (INE, divisiones CAE 18 y 14). Estas sí se publican por municipio,
# pero el último año disponible es 2022 (las subclases de arriba son de 2023)
carpeta_div = RAIZ / config["rutas"]["raw"] / f"ine_{ine['indicador_empresas_division']}"
respuestas_div = {ruta.stem.split("_")[-1]: recoleccion.leer_json(ruta) for ruta in sorted(carpeta_div.glob("*.json"))}

if respuestas_div:
    empresas_municipio = cleaning.empresas_por_municipio_division(respuestas_div, municipios, ine["divisiones_cae"])
    print("Municipios sin dato:", empresas_municipio.isna().any(axis=1).sum())
    display(empresas_municipio.sort_values(empresas_municipio.columns[1], ascending=False).head(10))
else:
    empresas_municipio = None
    print("Faltan las empresas por división del INE: ejecuta la sección 4 del notebook 04.")

Municipios sin dato: 0


,municipio,ine_vestuario_2022,ine_impresion_2022,ine_alojamiento_2022,ine_restauracion_2022,ine_salud_2022,ine_apoyo_social_con_alojamiento_2022,ine_apoyo_social_sin_alojamiento_2022
2,Coimbra,52,26,427,828,3767,29,61
39,Viseu,40,25,209,669,1580,9,31
4,Figueira da Foz,24,6,151,432,624,19,14
57,Covilhã,19,7,96,319,501,4,5
56,Castelo Branco,16,4,68,390,554,6,14
10,Oliveira do Hospital,15,4,67,104,117,4,6
47,Guarda,12,4,80,305,527,5,9
1,Cantanhede,12,8,43,212,320,7,10
52,Seia,12,5,100,140,181,4,3
22,Mangualde,10,1,33,92,110,0,3


## 4. Festivos municipales

**Cómo funciona**

- **Qué hace:** convierte las filas de Wikipedia en una tabla con el festivo de cada municipio y su fecha en 2026 y 2027.
- **Por qué así:** cada fila de Wikipedia trae varios municipios: `explode` crea una fila por municipio. Después se une con la lista de la región por el nombre normalizado. Las fechas: si la página trae la fecha de ese año, se usa; si es fija («13 de junho»), ese día; si depende de la Pascua («entre 23 de março e 26 de abril»), se calcula a partir de la Pascua de cada año. Resultado: 66 de 66.
- **Herramientas:** `apply` con `lambda`, `explode`, `merge`, `pd.to_datetime`, `str.contains`.
- **Nuevo (no visto en clase):** `re` (expresiones regulares) para leer las fechas y `dateutil.easter` para la fecha de la Pascua.

In [120]:
ruta_feriados = RUTA_RAW / "wikipedia_feriados_filas.json"

if ruta_feriados.exists():
    feriados = cleaning.limpiar_feriados(
        recoleccion.leer_json(ruta_feriados), municipios, config["wikipedia"]["anos_calendario"]
    )
    print("Municipios sin festivo encontrado:", feriados.loc[feriados["fecha_texto"].isna(), "municipio"].tolist())
    print(feriados["tipo_fecha"].value_counts())
else:
    feriados = None
    print("Faltan los festivos de Wikipedia: ejecuta primero el notebook 02.")

Municipios sin festivo encontrado: []
tipo_fecha
fija     56
movil    10
Name: count, dtype: int64


**Cómo funciona**

- **Qué hace:** cuenta los festivos municipales de cada mes de 2026.
- **Por qué así:** es la primera pista para la H4. `.dt.month` saca el mes de una fecha y `sort_index` ordena de enero a diciembre.
- **Herramientas:** `.dt.month`, `value_counts`, `sort_index`.

In [121]:
# Primera pista para la H4: festivos municipales por mes
if feriados is not None:
    primer_ano = config["wikipedia"]["anos_calendario"][0]
    display(feriados[f"fecha_{primer_ano}"].dt.month.value_counts().sort_index())

fecha_2026
1      1
2      1
3      2
4      7
5      7
6     14
7      9
8      7
9     13
10     3
11     2
Name: count, dtype: int64

## 5. Portal BASE: compras públicas de ropa

El notebook 02 deja los contratos de ropa y artículos promocionales de todo el país (un primer filtro amplio). Aquí `limpiar_base` hace siete pasos: quita duplicados, aplica un segundo filtro por categoría (solo lo que encaja con el catálogo), separa NIF y nombre del comprador y del proveedor, se queda con los municipios de la región, convierte fechas e importes y elimina los contratos sin fecha o sin precio.

Solo se muestran cifras: los nombres de los proveedores (la competencia) se guardan en un archivo interno, fuera de GitHub.

**Cómo funciona**

- **Qué hace:** limpia los contratos del Portal BASE en siete pasos y muestra cuántos quedan después de cada uno (12.350 → 12.292 → 7.265 → contratos de la región).
- **Por qué así:** 1) quita duplicados por `idcontrato`; 2) segundo filtro: solo lo que encaja con el catálogo (sin protección individual, calzado, ropa hospitalaria, lavandería ni artesanía), con los códigos y palabras de `config.yaml`; 3) separa NIF y nombre del comprador y del proveedor («510840744 - Município de Seia») y clasifica al comprador (cámara, junta, colegio, bomberos); 4) se queda con los municipios de la región por el lugar de ejecución; 5) convierte fecha, precio y CPV; 6) quita contratos sin fecha, sin precio o sin proveedor (los acuerdos marco a 0 €); 7) borra columnas que ya no hacen falta. Los `assert` comprueban precios positivos, proveedor identificado y municipios de la región.
- **Herramientas:** `drop_duplicates`, `str.extract`, `merge(how="inner")`, `pd.to_datetime`, `pd.to_numeric(errors="coerce")`, `dropna`, filtros.
- **Nuevo (no visto en clase):** identificar al proveedor por su NIF, porque el mismo nombre aparece escrito de varias formas («Lda» y «Lda.», mayúsculas).

In [122]:
rutas_base = [RUTA_RAW / "base" / f"contratos{ano}.xlsx" for ano in config["base"]["anos"]]
faltan = [ruta.name for ruta in rutas_base if not recoleccion.ruta_cache_ropa(ruta).exists()]

if not faltan:
    contratos = recoleccion.leer_contratos_ropa(rutas_base, config["base"]["prefijos_cpv"])
    df_base, registro_base = cleaning.limpiar_base(contratos, municipios, config["base"])
    display(pd.Series(registro_base, name="contratos"))
else:
    df_base = None
    print("Faltan los contratos filtrados de:", faltan, "- ejecuta primero el notebook 02.")

contratos2020.xlsx - leído del archivo filtrado
contratos2021.xlsx - leído del archivo filtrado
contratos2022.xlsx - leído del archivo filtrado
contratos2023.xlsx - leído del archivo filtrado
contratos2024.xlsx - leído del archivo filtrado
contratos2025.xlsx - leído del archivo filtrado


contratos_leidos                           12350
tras_quitar_duplicados                     12292
ropa_y_merchandising_portugal               7265
en_la_region                                 485
sin_fecha_precio_o_proveedor_eliminados        0
contratos_finales                            485
Name: contratos, dtype: int64

**Cómo funciona**

- **Qué hace:** cuenta los contratos por tipo de comprador y resume las compras de cámaras y juntas por año.
- **Por qué así:** las autarquías son el cliente objetivo. `isin` filtra varios valores a la vez y `agg` calcula varias medidas de una vez (contratos, gasto, mediana).
- **Herramientas:** `isin`, `groupby().agg()`, f-strings con `:,.0f`.

In [123]:
# Quién compra. El cliente objetivo son las autarquías: cámaras municipales y juntas de freguesia
if df_base is not None:
    display(df_base["tipo_comprador"].value_counts())
    autarquias = df_base[df_base["tipo_comprador"].isin(["camara", "junta"])]
    print(f"Autarquías: {len(autarquias)} contratos y {autarquias['precio'].sum():,.0f} € "
          f"entre {autarquias['ano'].min()} y {autarquias['ano'].max()}")
    display(autarquias.groupby("ano")["precio"].agg(contratos="count", gasto="sum", mediana="median").round(0))

tipo_comprador
camara     261
otro       158
escuela     62
junta        4
Name: count, dtype: int64

Autarquías: 265 contratos y 4,916,622 € entre 2020 y 2025


,contratos,gasto,mediana
ano,,,
2020,23,672187.0,12426.0
2021,26,543441.0,11965.0
2022,41,645767.0,11520.0
2023,51,703258.0,7996.0
2024,66,1152349.0,12103.0
2025,58,1199620.0,12565.0


**Cómo funciona**

- **Qué hace:** calcula contratos y gasto de las autarquías por mes de firma.
- **Por qué así:** primera pista para la H1. La prueba con simulación está en el 05.
- **Herramientas:** `groupby().agg()`, `round`.

In [124]:
# Primera pista para la H1: contratos y gasto de las autarquías por mes de firma
if df_base is not None:
    display(autarquias.groupby("mes")["precio"].agg(contratos="count", gasto="sum").round(0))

,contratos,gasto
mes,,
1,6,61961.0
2,23,389824.0
3,29,638822.0
4,14,294360.0
5,20,318656.0
6,21,489714.0
7,28,724758.0
8,36,571299.0
9,22,350004.0


**Cómo funciona**

- **Qué hace:** suma el gasto de cada proveedor, lo ordena de mayor a menor y calcula qué parte se llevan el mayor, los 3 y los 10 mayores.
- **Por qué así:** primera pista para la H2. Solo se muestran porcentajes: los nombres de la competencia no se publican.
- **Herramientas:** `groupby().sum()`, `sort_values`, `head(n).sum()`, `value_counts`.

In [125]:
# Primera pista para la H2: qué parte del gasto se llevan los mayores proveedores
# (solo porcentajes: los nombres de la competencia no se muestran)
if df_base is not None:
    gasto_proveedor = autarquias.groupby("proveedor_id")["precio"].sum().sort_values(ascending=False)
    print("Proveedores distintos:", len(gasto_proveedor))
    print("Proveedores con un solo contrato:", (autarquias["proveedor_id"].value_counts() == 1).sum())
    for n in [1, 3, 10]:
        print(f"Los {n} mayores se llevan el {gasto_proveedor.head(n).sum() / gasto_proveedor.sum():.1%} del gasto")

Proveedores distintos: 150
Proveedores con un solo contrato: 100
Los 1 mayores se llevan el 8.6% del gasto
Los 3 mayores se llevan el 14.9% del gasto
Los 10 mayores se llevan el 32.1% del gasto


**Cómo funciona**

- **Qué hace:** agrupa a los proveedores por lo que dice su nombre (gráfica, estampagem, publicidad…) y calcula su parte del gasto.
- **Por qué así:** mide cuánto ganan las empresas parecidas a la beira. «otro» significa que el nombre no dice a qué se dedica. `nunique` cuenta proveedores distintos.
- **Herramientas:** `groupby().agg()` con nombres, `nunique`, operaciones entre columnas.

In [126]:
# ¿Quién gana los contratos? Tipo de proveedor según su nombre (gráfica, estampagem, publicidad...).
# "otro": el nombre no dice a qué se dedica (por ejemplo, "Silva & Filhos, Lda"). Solo cifras, sin nombres.
if df_base is not None:
    por_tipo = autarquias.groupby("tipo_proveedor").agg(
        proveedores=("proveedor_id", "nunique"), contratos=("idcontrato", "count"), gasto=("precio", "sum")
    )
    por_tipo["pct_gasto"] = (por_tipo["gasto"] / por_tipo["gasto"].sum() * 100).round(1)
    display(por_tipo.sort_values("gasto", ascending=False).round(0))

,proveedores,contratos,gasto,pct_gasto
tipo_proveedor,,,,
otro,124,224,3999645.0,81.0
publicidad,7,10,259848.0,5.0
estampagem,3,6,190415.0,4.0
grafica,6,10,178482.0,4.0
personalizacion,6,7,137751.0,3.0
fardamento,3,6,130123.0,3.0
bordados,1,2,20359.0,0.0


**Cómo funciona**

- **Qué hace:** cuenta las autarquías que compran en dos años o más y, de ellas, las que repiten proveedor en años distintos.
- **Por qué así:** se agrupa por pareja comprador-proveedor y se cuenta en cuántos años distintos aparece. Primera pista para la H5; la simulación está en el 05.
- **Herramientas:** `groupby([...])`, `nunique`, `isin`, `reset_index`.

In [127]:
# Primera pista para la H5: ¿las autarquías que compran varios años repiten proveedor?
if df_base is not None:
    anos_por_comprador = autarquias.groupby("comprador_nif")["ano"].nunique()
    recurrentes = anos_por_comprador[anos_por_comprador >= 2].index
    # Pareja comprador-proveedor que aparece en dos años distintos o más
    anos_por_pareja = (
        autarquias[autarquias["comprador_nif"].isin(recurrentes)]
        .groupby(["comprador_nif", "proveedor_id"])["ano"].nunique()
    )
    repiten = anos_por_pareja[anos_por_pareja >= 2].reset_index()["comprador_nif"].nunique()
    print("Autarquías compradoras:", len(anos_por_comprador))
    print("Compran en dos años o más:", len(recurrentes))
    print("De esas, repiten proveedor en años distintos:", repiten)

Autarquías compradoras: 51
Compran en dos años o más: 32
De esas, repiten proveedor en años distintos: 21


**Cómo funciona**

- **Qué hace:** calcula el gasto por producto (camisetas, polos, ropa de trabajo, merchandising…) y el % de contratos por tipo de procedimiento.
- **Por qué así:** el producto sale del código CPV cuando es específico y, si no, de palabras de la descripción (`clasificar_producto`).
- **Herramientas:** `groupby().agg()`, `value_counts(normalize=True)`.

In [128]:
# Qué compran y por qué procedimiento
if df_base is not None:
    display(
        autarquias.groupby("producto")["precio"].agg(contratos="count", gasto="sum")
        .sort_values("gasto", ascending=False).round(0)
    )
    display((autarquias["tipoprocedimento"].value_counts(normalize=True) * 100).round(1))

,contratos,gasto
producto,,
merchandising,118,2592722.0
ropa de trabajo,94,1526432.0
otra ropa,25,409705.0
camisetas,12,170996.0
ropa deportiva,7,149237.0
polos,4,31385.0
sudaderas,2,23550.0
gorras,3,12594.0


tipoprocedimento
Consulta Prévia               46.0
Ajuste Direto Regime Geral    42.3
Concurso público              11.7
Name: proportion, dtype: float64

**Cómo funciona**

- **Qué hace:** guarda los contratos limpios (archivo interno, porque llevan nombres de proveedores) y resume contratos y gasto por municipio para la tabla final.
- **Por qué así:** `compras_por_municipio` separa el gasto de todos los compradores públicos y el de las autarquías.
- **Herramientas:** `to_csv`, `groupby().agg()`, `join`.

In [129]:
# Contratos limpios (interno: tienen nombres de proveedores) y compras por municipio para la tabla final
if df_base is not None:
    RUTA_INTERNO = RAIZ / config["rutas"]["interno"]
    RUTA_INTERNO.mkdir(parents=True, exist_ok=True)
    df_base.to_csv(RUTA_INTERNO / "contratos_ropa_region.csv", index=False)

    compras = cleaning.compras_por_municipio(df_base)
    print("Municipios con alguna compra publicada:", len(compras), "de", len(municipios))
    print("Con compras de autarquías:", compras["contratos_autarquias_ropa"].notna().sum())
else:
    compras = None

Municipios con alguna compra publicada: 55 de 66
Con compras de autarquías: 46


### Anuncios de concursos públicos de ropa en la región

Los concursos públicos se anuncian antes de adjudicarse y cualquier empresa puede presentarse. Sirven para saber cuántos hubo y cuántos días se dan para presentar una propuesta (y, más adelante, para vigilar los que estén abiertos).

**Cómo funciona**

- **Qué hace:** limpia los anuncios de concursos públicos de ropa de la región y calcula los días que dan para presentar propuesta.
- **Por qué así:** se quedan solo los anuncios originales (sin prórrogas ni rectificaciones), se reutiliza el filtro de ropa de los contratos (renombrando columnas) y el municipio sale del NIF del comprador o de su nombre, porque los anuncios no traen el lugar de ejecución.
- **Herramientas:** `rename`, `assign`, `map`, `mode`, bucle con `str.contains`.

In [130]:
rutas_anuncios = [RUTA_RAW / "base_anuncios" / f"anuncios{ano}.xlsx" for ano in config["base"]["anos"]]
rutas_anuncios = [ruta for ruta in rutas_anuncios if ruta.exists()]

if rutas_anuncios and df_base is not None:
    anuncios = recoleccion.leer_anuncios_ropa(rutas_anuncios, config["base"]["prefijos_cpv"])
    df_anuncios, registro_anuncios = cleaning.limpiar_anuncios(anuncios, df_base, municipios, config["base"])
    display(pd.Series(registro_anuncios, name="anuncios"))
    display(df_anuncios.groupby("ano").size().rename("concursos"))
    display(df_anuncios["tipo_comprador"].value_counts())
    print("Días para presentar propuesta (mediana):", df_anuncios["dias_para_propuestas"].median())
    df_anuncios.to_csv(RAIZ / config["rutas"]["interno"] / "anuncios_ropa_region.csv", index=False)
else:
    df_anuncios = None
    print("Faltan los anuncios: ejecuta la sección 5 del notebook 04.")

anuncios_leidos                  1240
anuncios_de_procedimiento        1056
ropa_y_merchandising_portugal     625
en_la_region                      110
anuncios_finales                  110
Name: anuncios, dtype: int64

ano
2020    16
2021    23
2022    14
2023    27
2024    11
2025    19
Name: concursos, dtype: int64

tipo_comprador
otro      96
camara    14
Name: count, dtype: int64

Días para presentar propuesta (mediana): 25.0


## 6. OpenStreetMap: competencia, empresas cliente y gráficas (opcional)

Salen del notebook 04. Si no se ha ejecutado, esta parte se salta y la tabla final se hace sin ellas.
OpenStreetMap solo registra una parte de los negocios: estos números son un mínimo, no el total.

**Cómo funciona**

- **Qué hace:** limpia los talleres y tiendas de la competencia que guardó el notebook 04 y los cuenta por tipo.
- **Por qué así:** `limpiar_negocios_osm` reutiliza los pasos de `limpiar_osm` (sin nombre, normalizar, contactos, duplicados). `reindex(fill_value=0)` muestra también los tipos con 0: OpenStreetMap no tiene ninguna bordadora ni imprenta en la región, y eso también es un resultado.
- **Herramientas:** `reindex`, `value_counts`, funciones reutilizadas.

In [131]:
# Competencia: talleres y tiendas registrados en OpenStreetMap (solo archivos guardados)
competencia_osm = recoleccion.recoger_osm_region(
    municipios, config["overpass"], config["overpass"]["etiquetas_competencia"], RUTA_RAW,
    solo_cache=True, subcarpeta="overpass_competencia",
)
if competencia_osm:
    df_competencia, registro_competencia = cleaning.limpiar_negocios_osm(
        competencia_osm, cleaning.TIPOS_COMPETENCIA, config["limpieza"]["licencia_osm"]
    )
    display(pd.Series(registro_competencia, name="filas"))
    # reindex muestra también los tipos sin ningún registro (con 0)
    tipos = [tipo for _, _, tipo in cleaning.TIPOS_COMPETENCIA]
    display(df_competencia["tipo"].value_counts().reindex(tipos, fill_value=0))
else:
    df_competencia = None
    print("No hay datos de competencia: ejecuta el notebook 04.")

overpass_competencia: 66 de 66 municipios leídos de los archivos guardados


filas_iniciales          476
sin_nombre_eliminadas     47
duplicados_eliminados     23
filas_finales            406
Name: filas, dtype: int64

tipo
bordado             0
imprenta            0
copisteria         19
tienda_deporte     29
tienda_ropa       358
Name: count, dtype: int64

**Cómo funciona**

- **Qué hace:** limpia los restaurantes, cafés, hoteles e instituciones sociales (posibles clientes de la oferta Trabajo) y calcula cuántos tienen email.
- **Por qué así:** mismas funciones que la competencia, con otra lista de tipos.
- **Herramientas:** `value_counts`, `notna().mean()`.

In [132]:
# Empresas como clientes (oferta Trabajo): restaurantes, cafés, hoteles e instituciones sociales
trabajo_osm = recoleccion.recoger_osm_region(
    municipios, config["overpass"], config["overpass"]["etiquetas_trabajo"], RUTA_RAW,
    solo_cache=True, subcarpeta="overpass_trabajo",
)
if trabajo_osm:
    df_trabajo, registro_trabajo = cleaning.limpiar_negocios_osm(
        trabajo_osm, cleaning.TIPOS_TRABAJO, config["limpieza"]["licencia_osm"]
    )
    display(pd.Series(registro_trabajo, name="filas"))
    display(df_trabajo["tipo"].value_counts())
    print(f"Con email público: {df_trabajo['email'].notna().mean():.1%}")
else:
    df_trabajo = None
    print("No hay datos de empresas: ejecuta el notebook 04.")

overpass_trabajo: 66 de 66 municipios leídos de los archivos guardados


filas_iniciales          3610
sin_nombre_eliminadas     457
duplicados_eliminados      90
filas_finales            3063
Name: filas, dtype: int64

tipo
restaurante           1479
cafe                  1110
hotel                  283
institucion_social     191
Name: count, dtype: int64

Con email público: 4.2%


**Cómo funciona**

- **Qué hace:** limpia los negocios encontrados por el nombre (gráfica, estampagem, bordados…) y los clasifica por tipo.
- **Por qué así:** con `por_nombre=True` el tipo sale del nombre normalizado y se quitan los que no encajan en ningún tipo.
- **Herramientas:** funciones reutilizadas, `value_counts`.

In [133]:
# Gráficas, estampagem y bordados buscados por nombre (solo archivos guardados)
graficas_osm = recoleccion.recoger_osm_region(
    municipios, config["overpass"], config["overpass"]["patron_graficas"], RUTA_RAW,
    solo_cache=True, subcarpeta="overpass_graficas",
)
if graficas_osm:
    df_graficas, registro_graficas = cleaning.limpiar_negocios_osm(
        graficas_osm, cleaning.TIPOS_GRAFICAS, config["limpieza"]["licencia_osm"], por_nombre=True
    )
    display(pd.Series(registro_graficas, name="filas"))
    display(df_graficas["tipo"].value_counts())
else:
    df_graficas = None
    print("No hay búsqueda de gráficas: ejecuta la sección 3 del notebook 04.")

overpass_graficas: 57 de 66 municipios leídos de los archivos guardados
   Sin archivo guardado para 9 municipios: ['Almeida', 'Guarda', 'Pinhel', 'Sabugal', 'Castelo Branco', 'Covilhã', 'Fundão', 'Idanha-a-Nova', 'Penamacor']


filas_iniciales          12
sin_nombre_eliminadas     0
sin_tipo_eliminadas       0
duplicados_eliminados     0
filas_finales            12
Name: filas, dtype: int64

tipo
grafica       6
estampagem    2
publicidad    2
fardamento    1
bordados      1
Name: count, dtype: int64

**Cómo funciona**

- **Qué hace:** guarda las tablas internas (nombres de competidores y contactos de empresas) y prepara un conteo por municipio de cada recogida.
- **Por qué así:** `contar_por_municipio` usa `pivot_table` para tener una columna por tipo. Un municipio descargado sin negocios queda con 0; uno no descargado queda vacío, porque no saber no es lo mismo que tener 0.
- **Herramientas:** listas, `if`, `to_csv`, `pivot_table`, `reindex`.

In [134]:
# Archivos internos (nombres de competidores y contactos de empresas) y conteos por municipio
RUTA_INTERNO = RAIZ / config["rutas"]["interno"]
RUTA_INTERNO.mkdir(parents=True, exist_ok=True)
RUTA_CLEAN.mkdir(parents=True, exist_ok=True)

conteos = []
if empresas_municipio is not None:
    conteos.append(empresas_municipio)
if df_competencia is not None:
    df_competencia.to_csv(RUTA_INTERNO / "competencia_osm.csv", index=False)
    conteos.append(cleaning.contar_por_municipio(
        df_competencia, "competencia", cleaning.TIPOS_COMPETENCIA, list(competencia_osm)
    ))
if df_trabajo is not None:
    df_trabajo.to_csv(RUTA_CLEAN / "leads_empresas.csv", index=False)  # interno: en .gitignore
    conteos.append(cleaning.contar_por_municipio(
        df_trabajo, "trabajo", cleaning.TIPOS_TRABAJO, list(trabajo_osm)
    ))
if df_graficas is not None:
    df_graficas.to_csv(RUTA_INTERNO / "graficas_osm.csv", index=False)
    conteos.append(cleaning.contar_por_municipio(
        df_graficas, "graficas", cleaning.TIPOS_GRAFICAS, list(graficas_osm)
    ))
print("Tablas de conteo para la tabla final:", len(conteos))

Tablas de conteo para la tabla final: 4


## 7. Tabla por municipio y archivos limpios

**Cómo funciona**

- **Qué hace:** quita las entidades excluidas (conflicto de intereses), une todo en una tabla por municipio y guarda `municipios.csv` y `leads_organizaciones.csv`.
- **Por qué así:** `resumen_municipios` parte de los 66 municipios de `config.yaml` y añade con `merge(how="left")` las organizaciones por segmento (`pivot_table`), la población, el festivo, la competencia, las compras y los conteos. Empezar por la lista completa garantiza que ningún municipio se pierde; el `assert` final lo comprueba. Las exclusiones se aplican aquí, después de añadir las juntas, para que no vuelvan a entrar.
- **Herramientas:** `merge`, `pivot_table`, `fillna(0)` solo donde 0 significa «ninguno», `astype("Int64")` (entero que admite vacíos), `to_csv`.

In [135]:
# Entidades a las que la marca no puede vender (conflicto de intereses): lista interna, fuera de GitHub
ruta_exclusiones = RAIZ / config["rutas"]["interno"] / "exclusiones.csv"
if ruta_exclusiones.exists():
    df_osm, excluidas = cleaning.aplicar_exclusiones(df_osm, pd.read_csv(ruta_exclusiones))
    print("Organizaciones excluidas:", excluidas)

tabla_municipios = cleaning.municipios_de_config(config["region"])
resumen = cleaning.resumen_municipios(
    tabla_municipios, df_osm[df_osm["origen"] == "OpenStreetMap"], poblacion, feriados, empresas, compras, conteos
)

RUTA_CLEAN.mkdir(parents=True, exist_ok=True)
resumen.to_csv(RUTA_CLEAN / "municipios.csv", index=False)
df_osm.to_csv(RUTA_CLEAN / "leads_organizaciones.csv", index=False)  # interno: en .gitignore

orden = "organizaciones_por_1000_hab" if poblacion is not None else "total_organizaciones"
resumen.sort_values(orden, ascending=False).head(10)

Organizaciones excluidas: 1


,municipio,distrito,piloto,asociacion,autarquia,bomberos,club,escuela,instalacion_deportiva,total_organizaciones,...,trabajo_hotel,trabajo_institucion_social,trabajo_total,graficas_bordados,graficas_estampagem,graficas_grafica,graficas_personalizacion,graficas_fardamento,graficas_publicidad,graficas_total
11,Pampilhosa da Serra,Coimbra,False,0,4,3,0,4,2,13,...,2,2,20,0,0,0,0,0,0,0
0,Arganil,Coimbra,True,9,14,3,1,10,6,43,...,4,5,62,0,0,1,0,0,0,1
17,Armamar,Viseu,False,2,10,1,0,3,3,19,...,6,1,21,0,0,0,0,0,0,0
48,Manteigas,Guarda,False,2,1,1,0,3,3,10,...,12,3,28,0,0,0,0,0,0,0
38,Vila Nova de Paiva,Viseu,False,1,3,1,0,6,2,13,...,1,7,25,0,0,0,0,0,0,0
5,Góis,Coimbra,False,3,3,2,0,3,0,11,...,0,3,17,0,0,0,0,0,0,0
45,Fornos de Algodres,Guarda,False,0,7,1,0,4,0,12,...,2,1,15,0,0,0,0,0,0,0
41,Aguiar da Beira,Guarda,False,1,4,1,0,5,2,13,...,2,3,11,0,0,0,0,0,0,0
60,Oleiros,Castelo Branco,False,1,3,1,1,4,2,12,...,1,1,19,0,0,0,0,0,0,0
51,Sabugal,Guarda,False,2,7,4,1,10,1,25,...,3,4,34,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
